# MoE fail-case metrics — saturation / co-activation / token-count

Reproduces the 11 synthetic router-failure scenarios from the colleague's
notebook (`luukkonenr/moe_metric_examples/dashboard.ipynb`) and runs the
metrics **we** use for the OLMoE analysis on each one:

* **token count / expert** — `compute_moe_metrics_hf._save_token_counts`
* **expert co-activation** — adjacent-rank, `_aggregate_coactivation`
* **router saturation vs final** — top-k overlap, `aggregate_hf_revisions.compute_router_saturation`

plus the colleague's scalar metrics (cv / entropy / max-prob / logit-spread / dead) for cross-reference.

**Pseudo-time via `severity`.** Saturation is temporal (overlap with the *final*
checkpoint), so we treat each scenario's `severity` knob as pseudo-time: with a
fixed token-noise seed we sweep severity 0→1 and compare every step to the
severity=1 endpoint (curve ends at 100%, like the real plots).

**Structural finding.** Top-k-overlap saturation is *invariant to logit scaling*,
so it is blind to `untrained_router`, `saturated_balanced`, `logit_explosion` —
exactly the pure-scaling failures the per-token entropy/max-prob metrics catch.

In [ ]:
"""Setup: import the shared fail-case module (source of truth, also runnable headless)."""
import os, sys
_HERE = os.path.abspath(os.path.dirname('__file__') or '.')
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

import matplotlib.pyplot as plt
import failcase_metrics as fc

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})

# OLMoE-like routing config (colleague used E=16, k=1).
E, TOPK, N, SEED = fc.E_DEFAULT, fc.TOPK_DEFAULT, fc.N_DEFAULT, fc.SEED_DEFAULT
print(f'E={E} top_k={TOPK} N={N} seed={SEED}  scenarios={len(fc.SCENARIOS)}')

In [ ]:
"""Run the severity sweep for every scenario."""
results = fc.run_all(E=E, N=N, topk=TOPK, seed=SEED)
print(fc.summary_table(results, E))

## Per-scenario panels

For each failure: **token count/expert** · **co-activation (top-32)** ·
**saturation vs final (pseudo-time)** · **scalar router-health**.

In [ ]:
"""Render all scenarios inline."""
for r in results:
    fig = fc.plot_scenario(r, E, TOPK)
    plt.show()

## Reading the table

* `sat_range` ≈ 0 ⇒ saturation cannot see the failure (scale-only modes:
  `untrained_router`, `saturated_balanced`, `logit_explosion`).
* Load failures (`full/partial_collapse`, `dead_experts`, `bimodal`, `heavy_tail`)
  move `cv%`, `dead`, and produce a rising saturation trajectory.
* `bias_drift` only moves `logit_spread` — load + saturation stay flat.
* `premature_specialization` only moves `max_prob` (sharp per-token, balanced load).

In [ ]:
"""Tweak the config and re-run (e.g. colleague's E=16, k=1)."""
# results16 = fc.run_all(E=16, N=4096, topk=1, seed=0)
# print(fc.summary_table(results16, 16))
# for r in results16: fc.plot_scenario(r, 16, 1); plt.show()